In [ ]:
# ============================================================
# DAY 31 — BIG REAL-WORLD RAG SYSTEM
# ============================================================
#
# This program demonstrates:
#
# 1. Document creation
# 2. Chunking
# 3. TF-IDF embeddings
# 4. Vector database
# 5. Dense retrieval
# 6. Keyword retrieval
# 7. Hybrid retrieval
# 8. Metadata filtering
# 9. Re-ranking
# 10. Recall@1
# 11. Recall@3
# 12. MRR
# 13. RAG
# 14. Grounding / "I don't know"
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import re
import numpy as np
import chromadb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. KNOWLEDGE BASE
# ============================================================
#
# Imagine this is a small company's internal knowledge base.
#
# In a real application these could come from:
#
# PDFs
# Word documents
# websites
# databases
# company notes
#
# ============================================================

DOCUMENTS = [

    {
        "id": "ml_01",
        "title": "Machine Learning Basics",
        "topic": "machine_learning",
        "text": """
        Machine learning allows computers to learn patterns from data
        instead of being explicitly programmed with every rule.
        A machine learning model receives examples during training
        and learns relationships that can be used to make predictions
        on new data.
        """
    },

    {
        "id": "ml_02",
        "title": "Training and Testing",
        "topic": "machine_learning",
        "text": """
        A dataset is commonly divided into training data and test data.
        Training data is used to learn the model parameters.
        Test data is kept separate so that we can estimate how well
        the trained model performs on previously unseen examples.
        """
    },

    {
        "id": "ml_03",
        "title": "Overfitting",
        "topic": "machine_learning",
        "text": """
        Overfitting happens when a machine learning model learns the
        training examples too closely, including accidental details
        and noise. An overfitted model may perform very well on training
        data but perform poorly on new unseen data.
        """
    },

    {
        "id": "ml_04",
        "title": "Regularization",
        "topic": "machine_learning",
        "text": """
        Regularization is a technique used to reduce overfitting.
        It adds a penalty for overly complex models. Common examples
        include L1 regularization and L2 regularization.
        """
    },

    {
        "id": "ml_05",
        "title": "Cross Validation",
        "topic": "machine_learning",
        "text": """
        Cross validation evaluates a model using several different
        train and validation splits. In five-fold cross validation,
        the dataset is divided into five parts and each part is used
        as the validation set once.
        """
    },

    {
        "id": "ml_06",
        "title": "Random Forest",
        "topic": "trees",
        "text": """
        A random forest is an ensemble machine learning algorithm
        that combines many decision trees. Each tree makes a prediction
        and the forest combines the predictions to produce a final result.
        Random forests are often useful because combining many trees
        can make the model more robust than a single decision tree.
        """
    },

    {
        "id": "ml_07",
        "title": "Decision Trees",
        "topic": "trees",
        "text": """
        A decision tree makes predictions by repeatedly splitting data
        according to features. Internal nodes represent decisions,
        branches represent possible outcomes, and leaf nodes contain
        the final prediction.
        """
    },

    {
        "id": "ml_08",
        "title": "Gradient Descent",
        "topic": "optimization",
        "text": """
        Gradient descent is an optimization algorithm used to reduce
        a model's loss. It calculates the direction in which the loss
        changes and moves the model parameters in the opposite direction.
        The learning rate controls the size of each update.
        """
    },

    {
        "id": "ml_09",
        "title": "Learning Rate",
        "topic": "optimization",
        "text": """
        The learning rate determines how large each update is during
        optimization. A learning rate that is too large can cause
        training to jump around or miss a good solution. A learning
        rate that is too small can make training very slow.
        """
    },

    {
        "id": "ml_10",
        "title": "Neural Networks",
        "topic": "deep_learning",
        "text": """
        A neural network contains layers of connected artificial neurons.
        Input data passes through one or more hidden layers before
        producing an output. During training, the network adjusts its
        weights to reduce prediction error.
        """
    },

    {
        "id": "ml_11",
        "title": "Embeddings",
        "topic": "nlp",
        "text": """
        An embedding represents text as a vector of numbers.
        Similar pieces of text can have similar vectors.
        Embeddings allow computers to compare text using mathematical
        similarity rather than only exact word matching.
        """
    },

    {
        "id": "ml_12",
        "title": "RAG",
        "topic": "nlp",
        "text": """
        Retrieval Augmented Generation, commonly called RAG,
        combines information retrieval with a language model.
        The system first retrieves relevant documents and then
        gives those documents to a language model as context.
        The model uses the retrieved context to generate an answer.
        """
    },

    {
        "id": "ml_13",
        "title": "Chunking",
        "topic": "nlp",
        "text": """
        Chunking divides a large document into smaller pieces.
        Smaller chunks make retrieval more precise because the system
        can return the part of a document that is most relevant to
        a particular question.
        """
    },

    {
        "id": "ml_14",
        "title": "Vector Databases",
        "topic": "nlp",
        "text": """
        A vector database stores vectors and allows applications
        to search for vectors that are similar to a query vector.
        This is useful for semantic search and retrieval augmented
        generation systems.
        """
    },

    {
        "id": "ml_15",
        "title": "Prompt Grounding",
        "topic": "nlp",
        "text": """
        Grounding tells a language model to base its answer on
        retrieved context instead of inventing information.
        A useful grounding instruction is: answer using only the
        provided context and say you do not know when the context
        does not contain the answer.
        """
    },

    {
        "id": "ml_16",
        "title": "Transformers",
        "topic": "deep_learning",
        "text": """
        Transformers are neural network architectures that use
        attention mechanisms to process relationships between tokens.
        They are widely used for modern natural language processing.
        """
    },

]


# ============================================================
# 2. CHUNKING
# ============================================================

def chunk_text(text, size=40, overlap=8):

    """
    Split a document into overlapping chunks.

    size:
        Maximum number of words in each chunk.

    overlap:
        Number of words shared between neighboring chunks.
    """

    words = text.split()

    if len(words) <= size:
        return [text.strip()]

    step = size - overlap

    chunks = []

    for start in range(0, len(words), step):

        end = start + size

        chunk = words[start:end]

        if chunk:
            chunks.append(
                " ".join(chunk)
            )

    return chunks


# ============================================================
# 3. BUILD CHUNKS
# ============================================================

def build_chunks():

    chunks = []

    chunk_number = 0

    for document in DOCUMENTS:

        pieces = chunk_text(
            document["text"],
            size=40,
            overlap=8
        )

        for piece in pieces:

            chunks.append({

                "id": f"chunk_{chunk_number}",

                "document_id": document["id"],

                "title": document["title"],

                "topic": document["topic"],

                "text": piece

            })

            chunk_number += 1

    return chunks


# ============================================================
# 4. CREATE TF-IDF EMBEDDINGS
# ============================================================

def create_embeddings(chunks):

    texts = [
        chunk["text"]
        for chunk in chunks
    ]

    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    embeddings = vectorizer.fit_transform(
        texts
    )

    return vectorizer, embeddings


# ============================================================
# 5. CREATE CHROMA DATABASE
# ============================================================

def create_vector_database(chunks, embeddings):

    client = chromadb.Client()

    collection = client.get_or_create_collection(
        name="day31_rag"
    )

    ids = [
        chunk["id"]
        for chunk in chunks
    ]

    documents = [
        chunk["text"]
        for chunk in chunks
    ]

    metadatas = [

        {
            "title": chunk["title"],
            "topic": chunk["topic"],
            "document_id": chunk["document_id"]
        }

        for chunk in chunks
    ]

    # Chroma expects normal Python lists.

    embedding_list = embeddings.toarray().tolist()

    collection.add(

        ids=ids,

        documents=documents,

        metadatas=metadatas,

        embeddings=embedding_list

    )

    return collection


# ============================================================
# 6. DENSE RETRIEVAL
# ============================================================

def dense_search(
    question,
    vectorizer,
    embeddings,
    chunks,
    k=3,
    topic=None
):

    # Convert question into vector.

    question_vector = vectorizer.transform(
        [question]
    )

    # Calculate similarity between question
    # and every chunk.

    scores = cosine_similarity(
        question_vector,
        embeddings
    )[0]

    results = []

    for index, score in enumerate(scores):

        chunk = chunks[index]

        # Optional metadata filter.

        if topic is not None:

            if chunk["topic"] != topic:
                continue

        results.append({

            "index": index,

            "score": float(score),

            "chunk": chunk

        })

    # Highest score first.

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:k]


# ============================================================
# 7. KEYWORD SEARCH
# ============================================================

def keyword_search(
    question,
    chunks,
    k=3
):

    question_words = set(
        re.findall(
            r"\b[a-zA-Z]+\b",
            question.lower()
        )
    )

    results = []

    for index, chunk in enumerate(chunks):

        text_words = set(
            re.findall(
                r"\b[a-zA-Z]+\b",
                chunk["text"].lower()
            )
        )

        if not question_words:

            score = 0

        else:

            common_words = (
                question_words & text_words
            )

            score = (
                len(common_words)
                /
                len(question_words)
            )

        results.append({

            "index": index,

            "score": float(score),

            "chunk": chunk

        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:k]


# ============================================================
# 8. HYBRID SEARCH
# ============================================================

def hybrid_search(
    question,
    vectorizer,
    embeddings,
    chunks,
    k=3,
    alpha=0.7
):

    dense = dense_search(
        question,
        vectorizer,
        embeddings,
        chunks,
        k=len(chunks)
    )

    keyword = keyword_search(
        question,
        chunks,
        k=len(chunks)
    )

    dense_scores = {
        item["index"]: item["score"]
        for item in dense
    }

    keyword_scores = {
        item["index"]: item["score"]
        for item in keyword
    }

    results = []

    for index, chunk in enumerate(chunks):

        dense_score = dense_scores.get(
            index,
            0
        )

        keyword_score = keyword_scores.get(
            index,
            0
        )

        hybrid_score = (

            alpha * dense_score

            +

            (1 - alpha) * keyword_score

        )

        results.append({

            "index": index,

            "score": hybrid_score,

            "dense_score": dense_score,

            "keyword_score": keyword_score,

            "chunk": chunk

        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:k]


# ============================================================
# 9. RE-RANKING
# ============================================================

def rerank(question, results):

    """
    Simple second-stage re-ranking.

    This is NOT a large language model.

    We are demonstrating the idea:

        first search
              ↓
        candidate results
              ↓
        second scoring step
              ↓
        final ranking
    """

    question_words = set(
        re.findall(
            r"\b[a-zA-Z]+\b",
            question.lower()
        )
    )

    reranked = []

    for result in results:

        text_words = set(
            re.findall(
                r"\b[a-zA-Z]+\b",
                result["chunk"]["text"].lower()
            )
        )

        common = (
            question_words &
            text_words
        )

        exact_match_score = (

            len(common)
            /
            max(len(question_words), 1)

        )

        original_score = result["score"]

        final_score = (

            0.7 * original_score

            +

            0.3 * exact_match_score

        )

        new_result = result.copy()

        new_result["rerank_score"] = (
            final_score
        )

        reranked.append(
            new_result
        )

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

def print_results(
    results,
    title="SEARCH RESULTS"
):

    print()
    print("=" * 70)

    print(title)

    print("=" * 70)

    for position, result in enumerate(
        results,
        start=1
    ):

        chunk = result["chunk"]

        print()

        print(
            f"#{position}"
        )

        print(
            "Title:",
            chunk["title"]
        )

        print(
            "Topic:",
            chunk["topic"]
        )

        print(
            "Score:",
            round(
                result["score"],
                3
            )
        )

        if "rerank_score" in result:

            print(
                "Re-rank score:",
                round(
                    result["rerank_score"],
                    3
                )
            )

        print(
            "Text:",
            chunk["text"]
        )


# ============================================================
# 11. EVALUATION DATA
# ============================================================
#
# Each question has the ID of the chunk/document
# that should contain the answer.
#
# ============================================================

EVALUATION = [

    (
        "What is overfitting?",
        "ml_03"
    ),

    (
        "How can we reduce overfitting?",
        "ml_04"
    ),

    (
        "What does a random forest do?",
        "ml_06"
    ),

    (
        "What is gradient descent?",
        "ml_08"
    ),

    (
        "What does the learning rate control?",
        "ml_09"
    ),

    (
        "What are embeddings?",
        "ml_11"
    ),

    (
        "What is RAG?",
        "ml_12"
    ),

    (
        "Why do we chunk documents?",
        "ml_13"
    ),

    (
        "What does a vector database do?",
        "ml_14"
    ),

    (
        "What is grounding?",
        "ml_15"
    ),

]


# ============================================================
# 12. CHECK WHETHER RESULT IS CORRECT
# ============================================================

def contains_correct_document(
    result,
    expected_document_id
):

    return (
        result["chunk"]["document_id"]
        ==
        expected_document_id
    )


# ============================================================
# 13. RECALL@K
# ============================================================

def calculate_recall(
    vectorizer,
    embeddings,
    chunks,
    k
):

    correct = 0

    total = len(
        EVALUATION
    )

    for question, expected_id in EVALUATION:

        results = dense_search(

            question,

            vectorizer,

            embeddings,

            chunks,

            k=k

        )

        found = any(

            contains_correct_document(
                result,
                expected_id
            )

            for result in results

        )

        if found:

            correct += 1

    return correct / total


# ============================================================
# 14. MRR
# ============================================================

def calculate_mrr(
    vectorizer,
    embeddings,
    chunks
):

    reciprocal_ranks = []

    for question, expected_id in EVALUATION:

        results = dense_search(

            question,

            vectorizer,

            embeddings,

            chunks,

            k=len(chunks)

        )

        rank = None

        for position, result in enumerate(
            results,
            start=1
        ):

            if contains_correct_document(
                result,
                expected_id
            ):

                rank = position

                break

        if rank is None:

            reciprocal_ranks.append(
                0
            )

        else:

            reciprocal_ranks.append(
                1 / rank
            )

    return np.mean(
        reciprocal_ranks
    )


# ============================================================
# 15. EVALUATE RETRIEVER
# ============================================================

def evaluate_retriever(
    vectorizer,
    embeddings,
    chunks
):

    print()
    print("=" * 70)
    print("RETRIEVAL EVALUATION")
    print("=" * 70)

    recall_1 = calculate_recall(

        vectorizer,
        embeddings,
        chunks,
        k=1

    )

    recall_3 = calculate_recall(

        vectorizer,
        embeddings,
        chunks,
        k=3

    )

    mrr = calculate_mrr(

        vectorizer,
        embeddings,
        chunks

    )

    print()

    print(
        f"Recall@1 : {recall_1:.3f}"
    )

    print(
        f"Recall@3 : {recall_3:.3f}"
    )

    print(
        f"MRR      : {mrr:.3f}"
    )


# ============================================================
# 16. RAG PROMPT BUILDER
# ============================================================

def build_prompt(
    question,
    results
):

    context = "\n\n".join(

        result["chunk"]["text"]

        for result in results

    )

    prompt = f"""
You are a helpful AI assistant.

Answer the question using ONLY the context below.

If the context does not contain enough information
to answer the question, say:

"I don't know based on the provided documents."

Do not invent facts.

---------------- CONTEXT ----------------

{context}

---------------- QUESTION ----------------

{question}

---------------- ANSWER ----------------
"""

    return prompt


# ============================================================
# 17. FAKE LLM
# ============================================================
#
# We are not calling OpenAI here.
#
# This lets the complete RAG pipeline work offline.
#
# Later you can replace this function with a real LLM API.
#
# ============================================================

def fake_llm(
    question,
    results
):

    if not results:

        return (
            "I don't know based on "
            "the provided documents."
        )

    best = results[0]

    score = best["score"]

    if score < 0.05:

        return (
            "I don't know based on "
            "the provided documents."
        )

    return (
        "Based on the retrieved documents: "
        +
        best["chunk"]["text"]
    )


# ============================================================
# 18. COMPLETE RAG PIPELINE
# ============================================================

def rag(
    question,
    vectorizer,
    embeddings,
    chunks
):

    # --------------------------------------------------------
    # STEP 1 — RETRIEVE
    # --------------------------------------------------------

    results = hybrid_search(

        question,

        vectorizer,

        embeddings,

        chunks,

        k=5,

        alpha=0.7

    )


    # --------------------------------------------------------
    # STEP 2 — RE-RANK
    # --------------------------------------------------------

    results = rerank(

        question,

        results

    )


    # --------------------------------------------------------
    # STEP 3 — KEEP TOP 3
    # --------------------------------------------------------

    final_results = results[:3]


    # --------------------------------------------------------
    # STEP 4 — BUILD GROUNDED PROMPT
    # --------------------------------------------------------

    prompt = build_prompt(

        question,

        final_results

    )


    # --------------------------------------------------------
    # STEP 5 — GENERATE ANSWER
    # --------------------------------------------------------

    answer = fake_llm(

        question,

        final_results

    )


    return (
        final_results,
        prompt,
        answer
    )


# ============================================================
# 19. MAIN PROGRAM
# ============================================================

def main():

    print()
    print("=" * 70)
    print("           DAY 31 — COMPLETE RAG ENGINE")
    print("=" * 70)


    # --------------------------------------------------------
    # BUILD CHUNKS
    # --------------------------------------------------------

    print()
    print("1. Building chunks...")

    chunks = build_chunks()

    print(
        "Documents:",
        len(DOCUMENTS)
    )

    print(
        "Chunks:",
        len(chunks)
    )


    # --------------------------------------------------------
    # CREATE EMBEDDINGS
    # --------------------------------------------------------

    print()
    print("2. Creating TF-IDF embeddings...")

    vectorizer, embeddings = (
        create_embeddings(chunks)
    )

    print(
        "Embedding matrix shape:",
        embeddings.shape
    )


    # --------------------------------------------------------
    # CREATE VECTOR DATABASE
    # --------------------------------------------------------

    print()
    print("3. Creating Chroma vector database...")

    collection = create_vector_database(

        chunks,

        embeddings

    )

    print(
        "Vector database created."
    )

    print(
        "Stored vectors:",
        collection.count()
    )


    # --------------------------------------------------------
    # EVALUATION
    # --------------------------------------------------------

    evaluate_retriever(

        vectorizer,

        embeddings,

        chunks

    )


    # --------------------------------------------------------
    # DEMO QUESTION
    # --------------------------------------------------------

    question = (
        "How can we reduce overfitting?"
    )


    # --------------------------------------------------------
    # DENSE SEARCH
    # --------------------------------------------------------

    dense_results = dense_search(

        question,

        vectorizer,

        embeddings,

        chunks,

        k=3

    )

    print_results(

        dense_results,

        "DENSE SEARCH"

    )


    # --------------------------------------------------------
    # KEYWORD SEARCH
    # --------------------------------------------------------

    keyword_results = keyword_search(

        question,

        chunks,

        k=3

    )

    print_results(

        keyword_results,

        "KEYWORD SEARCH"

    )


    # --------------------------------------------------------
    # HYBRID SEARCH
    # --------------------------------------------------------

    hybrid_results = hybrid_search(

        question,

        vectorizer,

        embeddings,

        chunks,

        k=3,

        alpha=0.7

    )

    print_results(

        hybrid_results,

        "HYBRID SEARCH"

    )


    # --------------------------------------------------------
    # RE-RANK
    # --------------------------------------------------------

    reranked_results = rerank(

        question,

        hybrid_results

    )

    print_results(

        reranked_results,

        "RE-RANKED RESULTS"

    )


    # --------------------------------------------------------
    # RAG
    # --------------------------------------------------------

    final_results, prompt, answer = rag(

        question,

        vectorizer,

        embeddings,

        chunks

    )


    print()
    print("=" * 70)
    print("RAG SYSTEM")
    print("=" * 70)


    print()

    print(
        "Question:",
        question
    )


    print()

    print(
        "Retrieved context:"
    )

    for result in final_results:

        print(
            "-",
            result["chunk"]["title"]
        )


    print()

    print(
        "Generated answer:"
    )

    print(
        answer
    )


    # --------------------------------------------------------
    # SHOW PROMPT
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("GROUNDED PROMPT SENT TO LLM")
    print("=" * 70)

    print(
        prompt
    )


    # --------------------------------------------------------
    # METADATA FILTER DEMO
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("METADATA FILTER DEMO")
    print("=" * 70)


    filtered_results = dense_search(

        "What is attention?",

        vectorizer,

        embeddings,

        chunks,

        k=3,

        topic="deep_learning"

    )

    print_results(

        filtered_results,

        "ONLY DEEP LEARNING DOCUMENTS"

    )


    # --------------------------------------------------------
    # INTERACTIVE MODE
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("INTERACTIVE RAG")
    print("=" * 70)

    print()
    print(
        "Ask questions about the knowledge base."
    )

    print(
        "Type 'quit' to stop."
    )


    while True:

        print()

        user_question = input(
            "Question: "
        )


        if user_question.lower() == "quit":

            break


        if not user_question.strip():

            continue


        results, prompt, answer = rag(

            user_question,

            vectorizer,

            embeddings,

            chunks

        )


        print()

        print(
            "TOP RETRIEVED DOCUMENTS:"
        )


        for result in results:

            print(

                f"- {result['chunk']['title']} "
                f"(score={result['rerank_score']:.3f})"

            )


        print()

        print(
            "ANSWER:"
        )

        print(
            answer
        )


    # --------------------------------------------------------
    # FINISHED
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("DAY 31 COMPLETE")
    print("=" * 70)


# ============================================================
# START PROGRAM
# ============================================================

if __name__ == "__main__":

    main()


           DAY 31 — COMPLETE RAG ENGINE

1. Building chunks...
Documents: 16
Chunks: 19

2. Creating TF-IDF embeddings...
Embedding matrix shape: (19, 201)

3. Creating Chroma vector database...
Vector database created.
Stored vectors: 19

RETRIEVAL EVALUATION

Recall@1 : 0.900
Recall@3 : 0.900
MRR      : 0.907

DENSE SEARCH

#1
Title: Regularization
Topic: machine_learning
Score: 0.256
Text: Regularization is a technique used to reduce overfitting.
        It adds a penalty for overly complex models. Common examples
        include L1 regularization and L2 regularization.

#2
Title: Overfitting
Topic: machine_learning
Score: 0.142
Text: Overfitting happens when a machine learning model learns the
        training examples too closely, including accidental details
        and noise. An overfitted model may perform very well on training
        data but perform poorly on new unseen data.

#3
Title: Neural Networks
Topic: deep_learning
Score: 0.113
Text: A neural network contains layers

Question:  who is pm of nepal



TOP RETRIEVED DOCUMENTS:
- Gradient Descent (score=0.204)
- Chunking (score=0.204)
- Prompt Grounding (score=0.204)

ANSWER:
Based on the retrieved documents: Gradient descent is an optimization algorithm used to reduce
        a model's loss. It calculates the direction in which the loss
        changes and moves the model parameters in the opposite direction.
        The learning rate controls the size of each update.

